# Interactive Training Session

SageMaker **interactive training sessions** give you a live, client-driven
training loop over a hosted model: you submit low-level primitives
(`forward_backward`, `optim_step`, `forward`), checkpoint with
`save_weights_for_sampler` / `save_state` / `load_state`, and sample from a
checkpoint with
`sample` — all from your own Python process.

The SDK assigns operation sequence numbers for you (so calls can be pipelined)
and signs every request with your own AWS credentials (SigV4). You only need
standard `boto3`-resolvable credentials configured (env vars, profile, or an
attached role) — there is no separate API key or token to manage.

This notebook has two parts:
1. **Quick start** — the minimal end-to-end loop.
2. **Going further** — train on real text with a tokenizer, inspect metrics,
   checkpoint/resume, and sample.

## Prerequisites

Before running, you need:
- A SageMaker **model package group** to register checkpoints under.
- An **IAM execution role** SageMaker assumes to access your resources.
- Permissions on **your own identity** for the training-session actions, plus
  `iam:PassRole` for the execution role above, since these calls hand that role
  to the service.
- An **S3 output path** for checkpoints and results.
- AWS credentials + region configured for `boto3`.
- `pip install 'sagemaker-train[tokenizer]'` (the tokenizer is used for text prompts).

If you don't yet have a model package group, execution role, or bucket, the
**Optional: create prerequisites** section below shows how to create them.

In [ ]:
# Install the SDK (provides the `sagemaker.train.*` namespace) and the
# tokenizer used for text prompts. Restart the kernel after installing.
%pip install 'sagemaker-train[tokenizer]'

## Configuration

Update these values for your account before running.

In [ ]:
# Region to run in. The base models on offer differ by region; see
# "Find a base model" below.
REGION = "us-west-2"

# A name you choose for this session (any string):
TRAINING_SESSION_NAME = "<YOUR_TRAINING_SESSION_NAME>"

# Base model to fine-tune from — an id from list_base_models() (see "Find a
# base model" below). The SDK resolves it to the latest Hub model ARN in REGION.
# To pin a specific version, set a full Hub model ARN here instead.
BASE_MODEL = "<id from list_base_models()>"

# Resources in your account (create them in the next section if needed):
MODEL_PACKAGE_GROUP_ARN = (
    f"arn:aws:sagemaker:{REGION}:<ACCOUNT_ID>"
    ":model-package-group/<YOUR_MODEL_PACKAGE_GROUP_NAME>"
)
ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<YOUR_ROLE_NAME>"
OUTPUT_S3_PATH = "s3://<YOUR_BUCKET_NAME>/output/"

LORA_RANK = 8
LORA_ALPHA = 16

## Optional: create prerequisites

Skip this if you already have a model package group, execution role, and S3
bucket. These helper cells create them with `boto3`. Run only the ones you need,
then copy the resulting ARNs / paths into the **Configuration** cell above.

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
sts = boto3.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]

# 1) Model package group
MPG_NAME = "my-training-session-models"
try:
    resp = sm.create_model_package_group(ModelPackageGroupName=MPG_NAME)
    print("created:", resp["ModelPackageGroupArn"])
except sm.exceptions.ClientError as e:
    print("(may already exist)", e)

# 2) S3 bucket for outputs
# Named sagemaker-* so the managed policy's S3 permissions cover it.
BUCKET = f"sagemaker-training-session-{ACCOUNT_ID}-{REGION}"
try:
    kwargs = {"Bucket": BUCKET}
    if REGION != "us-east-1":
        kwargs["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3.create_bucket(**kwargs)
    print("created bucket:", BUCKET)
except Exception as e:
    print("(may already exist)", e)

print("\nSet in Configuration above:")
print(f'  MODEL_PACKAGE_GROUP_ARN = "arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:model-package-group/{MPG_NAME}"')
print(f'  OUTPUT_S3_PATH          = "s3://{BUCKET}/output/"')

The execution role needs a permissions policy allowing `sagemaker:CreateModelPackage`,
`sagemaker:DescribeHubContent`, `sagemaker:DescribeModelPackageGroup`, and S3
read/write on your bucket, plus a trust policy letting
`training-session.sagemaker.amazonaws.com` assume it (`sts:AssumeRole` +
`sts:TagSession`). See the package README for the exact JSON. Set `ROLE_ARN`
above to the role you create.

In [ ]:
# Create an execution role SageMaker assumes for the training session.
# Skip if you already have one — just set ROLE_ARN above.
import json

iam = boto3.client("iam")
ROLE_NAME = "SageMakerTrainingSessionRole"

trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "training-session.sagemaker.amazonaws.com"},
        "Action": ["sts:AssumeRole", "sts:TagSession"],
    }],
}
perms = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow",
         "Action": ["sagemaker:CreateModelPackage", "sagemaker:DescribeHubContent",
                    "sagemaker:DescribeModelPackageGroup"],
         "Resource": "*"},
        {"Effect": "Allow",
         "Action": ["s3:ListBucket", "s3:GetObject", "s3:PutObject"],
         "Resource": [f"arn:aws:s3:::{BUCKET}", f"arn:aws:s3:::{BUCKET}/*"]},
    ],
}
try:
    r = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))
    iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="training-session-access",
                        PolicyDocument=json.dumps(perms))
    print("created role:", r["Role"]["Arn"])
except iam.exceptions.EntityAlreadyExistsException:
    print("role already exists:", f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}")

print(f'\nSet:  ROLE_ARN = "arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"')

### Find a base model

`BASE_MODEL` is the model you fine-tune **from**. Pass an id from
`svc.list_base_models()` (listed below) and the SDK resolves it to the latest
Hub model ARN in your region. To pin a specific model version, pass that full
Hub model ARN as `BASE_MODEL` instead.

Checkpoints this session saves are registered as model package *versions* in
your model package group and are addressed by ARN, so there is no name to
choose for them.

In [ ]:
# Base models available to fine-tune in your region (pass one as BASE_MODEL above).
# Queries the SageMaker Public Hub in REGION and returns each model's own
# model id. Models are region-specific, so this can differ by region.
from sagemaker.train.training_session import ServiceClient

for model_id in ServiceClient(region=REGION, role_arn=ROLE_ARN).list_base_models():
    print(model_id)

---
# Part 1 — Quick start

## Create a Training Session

`create_lora_training_client` creates the session and blocks until it reaches
`Running`, then returns a `TrainingClient` bound to it.

In [ ]:
from sagemaker.train.training_session import (
    ServiceClient,
    AdamParams,
    Datum,
    LossFunction,
    ModelInput,
    SamplingParams,
    TensorData,
    TensorDtype,
)

svc = ServiceClient(
    region=REGION,
    role_arn=ROLE_ARN,
    model_package_group_arn=MODEL_PACKAGE_GROUP_ARN,
    s3_output_path=OUTPUT_S3_PATH,
)
tc = svc.create_lora_training_client(
    base_model=BASE_MODEL,
    training_session_name=TRAINING_SESSION_NAME,
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
)
print(f"Session running: {tc.training_session_name}")

## A minimal training step

Pass plain text and let the SDK tokenize it (it loads the base model's tokenizer
on first use). Submit a `forward_backward` then an `optim_step`; each call
returns a future you await with `.result()`.

In [ ]:
# cross_entropy needs per-token loss inputs: which token to predict at each
# position (target_tokens) and how much each position counts (weights).
# Both run the full length of the tokenized input.
tokens = tc.get_tokenizer().encode(
    "Reinforcement learning teaches an agent to maximize reward."
)

fb = tc.forward_backward(
    [
        Datum(
            model_input=ModelInput.from_ints(tokens),
            loss_fn_inputs={
                "target_tokens": TensorData(
                    data=tokens[1:] + [tokens[-1]],
                    dtype=TensorDtype.INT64,
                    shape=[len(tokens)],
                ),
                "weights": TensorData(
                    data=[1.0] * len(tokens),
                    dtype=TensorDtype.FLOAT32,
                    shape=[len(tokens)],
                ),
            },
        )
    ],
    loss_fn="cross_entropy",
)
print("forward_backward metrics:", fb.result().metrics)

optim = tc.optim_step(AdamParams(learning_rate=1e-4))
print("optim_step metrics:", optim.result().metrics)

---
# Part 2 — Going further

## Pipeline a real training loop over text

Operations are pipelined: submit `forward_backward` and `optim_step` for a step
without blocking, then await both. The SDK orders them via auto-assigned
sequence numbers. Swap `loss_fn` for any supported objective
(`cross_entropy`, `importance_sampling`, `ppo`, `cispo`).

In [ ]:
corpus = [
    "The capital of France is Paris.",
    "Gradient descent minimizes a loss function step by step.",
    "A transformer attends over all positions in a sequence.",
    "LoRA adapts a base model with a small number of trainable parameters.",
]

def to_datum(text):
    """Build one cross_entropy training datum from a line of text."""
    ids = tc.get_tokenizer().encode(text)
    return Datum(
        model_input=ModelInput.from_ints(ids),
        loss_fn_inputs={
            "target_tokens": TensorData(
                data=ids[1:] + [ids[-1]], dtype=TensorDtype.INT64, shape=[len(ids)]
            ),
            "weights": TensorData(
                data=[1.0] * len(ids), dtype=TensorDtype.FLOAT32, shape=[len(ids)]
            ),
        },
    )


for step, text in enumerate(corpus):
    fb = tc.forward_backward([to_datum(text)], loss_fn=LossFunction.CROSS_ENTROPY)
    opt = tc.optim_step(AdamParams(learning_rate=1e-4))
    fb_res, opt_res = fb.result(), opt.result()
    print(f"step {step}: {fb_res.metrics}  {text[:40]!r}")

## Inspect a forward-only pass

`forward` runs the model without a gradient update — useful for evaluating loss
or reading `loss_fn_outputs` (e.g. per-token logprobs) on held-out text.

In [ ]:
# A forward-only pass still needs the loss inputs its loss function requires,
# built the same way as the training pass above.
eval_tokens = tc.get_tokenizer().encode("Paris is the capital of France.")

fwd = tc.forward(
    [
        Datum(
            model_input=ModelInput.from_ints(eval_tokens),
            loss_fn_inputs={
                "target_tokens": TensorData(
                    data=eval_tokens[1:] + [eval_tokens[-1]],
                    dtype=TensorDtype.INT64,
                ),
                "weights": TensorData(data=[1.0] * len(eval_tokens)),
            },
        )
    ],
    loss_fn=LossFunction.CROSS_ENTROPY,
)
fwd_res = fwd.result()
print("eval metrics:", fwd_res.metrics)

## Checkpoint and resume

`save_weights_for_sampler` registers inference weights as a model package.
`save_state` additionally saves optimizer state so training can resume;
`load_state` restores it.

In [ ]:
save_state = tc.save_state()
model_package_arn = save_state.result().model_package_arn
print("save_state model_package_arn:", model_package_arn)

tc.load_state_with_optimizer(model_package_arn).result()
print("load_state_with_optimizer: success")

## Sample from the trained checkpoint

`save_weights_and_get_sampling_client` saves weights and returns a
`SamplingClient` bound to that checkpoint. `sample` returns a future. Use
`get_tokenizer()` to turn text into token IDs and to decode the result back to
text; you can also pass token IDs directly.

In [ ]:
sc = tc.save_weights_and_get_sampling_client()
tokenizer = sc.get_tokenizer()
prompt_tokens = tokenizer.encode("What is reinforcement learning?")

result = sc.sample(
    prompt=prompt_tokens,
    num_samples=1,
    sampling_params=SamplingParams(max_tokens=512, temperature=0.7),
).result()

for seq in result.sequences:
    print("text:", tokenizer.decode(seq.tokens))
    print("stop_reason:", seq.stop_reason)

## Resume from a checkpoint

After a notebook restart, resume from a `save_state` checkpoint in a new
session. `create_training_client_from_state_with_optimizer` restores weights
and optimizer state; `create_training_client_from_state` restores weights only.

In [ ]:
from sagemaker.train.training_session import ModelPackageKind

ckpt = svc.get_latest_model_package(
    ModelPackageKind.CHECKPOINT,
    model_package_group_arn=MODEL_PACKAGE_GROUP_ARN,
)
resumed_tc = svc.create_training_client_from_state_with_optimizer(
    model_package_arn=ckpt.model_package_arn,
)
print(f"Resumed in a new session: {resumed_tc.training_session_name}")

## Cleanup

Stop and delete every session this notebook created, so none keeps holding GPU capacity. `stop_if_active=True` stops a
running session and waits for a terminal state before deleting.

In [ ]:
# Both sessions this notebook created: the one from Part 1 and the one the
# resume section started. Leaving either running keeps billing GPU capacity.
# The resume section is optional, so its client may not exist. Collect the
# names first: naming a missing one here would raise before anything is
# deleted, leaving every session running.
names = [TRAINING_SESSION_NAME]
if "resumed_tc" in dir():
    names.append(resumed_tc.training_session_name)

for name in names:
    svc.delete_session(name, stop_if_active=True)
    print(f"Deleted session: {name}")

## Sweep leaked sessions (optional)

A safety net if a run failed before its cleanup cell and left sessions active.
This lists active sessions via `list_sessions()` and deletes them with
`delete_session(stop_if_active=True)`. It is **dry-run by default** — set
`DRY_RUN = False` to actually delete.

In [ ]:
DRY_RUN = True  # set to False to actually stop + delete

from sagemaker.train.training_session import SessionStatus

ACTIVE = {SessionStatus.CREATED, SessionStatus.SCHEDULED, SessionStatus.RUNNING}
active_sessions = [s for s in svc.list_sessions() if s.status in ACTIVE]

if not active_sessions:
    print("No active sessions to clean up.")
for s in active_sessions:
    name = s.training_session_name
    if DRY_RUN:
        print(f"[dry-run] would delete {name!r} (status={s.status})")
    else:
        svc.delete_session(name, stop_if_active=True)
        print(f"Deleted {name!r}")